In [ ]:
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install \
    "docling-core>=2.70,<3" \
    "docling-agent==0.1.0" \
    "mellea[litellm]==0.4.*" \
    "git+https://github.com/ibm-granite-community/utils.git"
! echo "::endgroup::"

# Chunkless RAG with Docling

In the previous tutorial in this series, you built a retrieval-augmented Q&A system over a `DoclingDocument` by **chunking** the document, **embedding** the chunks, and serving queries with vector top-k. That's the workhorse pattern most RAG stacks use, and for many problems it's the right call.

This tutorial throws both of those out.

We're going to ask the same kind of question against the same kind of document, but with **no chunker, no embedding model, and no vector store**. Instead, we'll let the LLM **navigate the document's own structural tree** — picking sections by reading their summaries, the way a human researcher skims a table of contents — and answer from whatever section it lands on.

This is what `docling-agent` calls a **chunkless RAG agent** (`DoclingRAGAgent`). It only works because Docling has already done the hard work of recovering real document structure from the source PDF. By the end of this notebook you'll know:

1. How chunkless navigation actually works (it's roughly 80 lines of loop, not magic).
2. How to run it on a pre-enriched `DoclingDocument`, and how to enrich your own.
3. When to reach for it instead of hybrid-chunking RAG — and when not to.

In [ ]:
import os
from pathlib import Path

from docling_core.types.doc.document import DoclingDocument
from docling_core.experimental.serializer.outline import OutlineFormat
from mellea.backends import model_ids

from docling_agent.agent.base_functions import create_document_outline
from docling_agent.agents import DoclingRAGAgent, DoclingEnrichingAgent
from ibm_granite_community.notebook_utils import get_env_var

# ---------------------------------------------------------------------------
# Pick a model + backend. Uncomment exactly ONE pairing below.
# All pairings go through Mellea, so you can plug in any other Mellea
# backend (vLLM, Watsonx, Bedrock, HuggingFace) the same way.
# ---------------------------------------------------------------------------

# Hosted via Replicate. Default for this notebook: Granite 4.0 H Small (32B).
# More capable than local 8B models on longer documents and doesn't need a GPU.
MODEL_ID = model_ids.IBM_GRANITE_4_HYBRID_SMALL
BACKEND = "replicate"

# Local Ollama alternative. Friendly on CPU-only machines but may struggle
# on long outlines. Requires: `ollama pull granite3.3:8b` and `ollama serve`.
# MODEL_ID = model_ids.IBM_GRANITE_3_3_8B
# BACKEND = "ollama"

# Hosted via Mellea's OpenAI backend. Set OPENAI_API_KEY before running.
# MODEL_ID = model_ids.OPENAI_GPT_5_1
# BACKEND = "openai"

if BACKEND == "replicate":
    # get_env_var reads Colab Secrets on Colab and os.environ locally.
    os.environ["REPLICATE_API_TOKEN"] = get_env_var("REPLICATE_API_TOKEN")

print(f"Using model: {MODEL_ID.hf_model_name} via {BACKEND}")

In [ ]:
# ---------------------------------------------------------------------------
# Backend routing. docling-agent's default `setup_local_session` hardcodes
# Ollama; this cell patches it to pick a backend based on the BACKEND global
# above. We patch every module that binds `setup_local_session` at import
# time (Python's `from X import Y` creates a separate reference). This
# keeps the docling-agent source untouched — everything lives in the notebook.
#
# docling-agent 0.1.0 imports `setup_local_session` in six modules:
#   - docling_agent.agent.rag           (exercised by DoclingRAGAgent)
#   - docling_agent.agent.enricher      (exercised by DoclingEnrichingAgent)
#   - docling_agent.agent.editor        (not used in this notebook)
#   - docling_agent.agent.extractor     (not used in this notebook)
#   - docling_agent.agent.orchestrator  (not used in this notebook)
#   - docling_agent.agent.writer        (not used in this notebook)
# We patch only the two this notebook exercises. If you extend the notebook
# to use editor/extractor/orchestrator/writer, add them to the loop below.
# Long-term fix: open an issue upstream requesting a supported backend
# override on the agent classes so this patch can be deleted entirely.
# ---------------------------------------------------------------------------
import importlib

from mellea import MelleaSession
from mellea.backends.ollama import OllamaModelBackend
from mellea.stdlib.components import Message
from mellea.stdlib.context import ChatContext

import docling_agent.agent_models as _am


def _make_backend(model_id):
    if BACKEND == "replicate":
        from mellea.backends.litellm import LiteLLMBackend
        # LiteLLM routes "replicate/<owner>/<model>" to Replicate's API
        # using the REPLICATE_API_TOKEN environment variable.
        return LiteLLMBackend(model_id=f"replicate/{model_id.hf_model_name}")
    if BACKEND == "openai":
        from mellea.backends.openai import OpenAIBackend
        return OpenAIBackend(model_id=model_id)
    return OllamaModelBackend(model_id=model_id)


def _patched_setup_local_session(*, model_id, system_prompt="You are a helpful assistant."):
    ctx = ChatContext().add(Message(role="system", content=system_prompt))
    m = MelleaSession(ctx=ctx, backend=_make_backend(model_id))
    return _am._LoggingSession(m)


_am.setup_local_session = _patched_setup_local_session
for modname in (
    "docling_agent.agent.rag",
    "docling_agent.agent.enricher",
):
    mod = importlib.import_module(modname)
    if hasattr(mod, "setup_local_session"):
        mod.setup_local_session = _patched_setup_local_session

print(f"setup_local_session patched -> {BACKEND}")

## How chunkless RAG works

Chunkless retrieval is a four-step loop:

```
                   ┌──────────────────────────────────────┐
                   │       Document outline               │
                   │    (per-section summaries)           │
                   └──────────────────────────────────────┘
                                   │
                                   ▼
 ┌──────────────────────────────────────────────────────────┐
 │ 1. SELECT  — LLM picks the most relevant unvisited       │
 │              section by reading the outline + query.     │
 └──────────────────────────────────────────────────────────┘
                                   │
                                   ▼
 ┌──────────────────────────────────────────────────────────┐
 │ 2. FETCH   — Pull the *full text* of that section's      │
 │              subtree from the DoclingDocument.           │
 └──────────────────────────────────────────────────────────┘
                                   │
                                   ▼
 ┌──────────────────────────────────────────────────────────┐
 │ 3. ATTEMPT — LLM tries to answer from the section text.  │
 │              Returns {can_answer: bool, response: str}.  │
 └──────────────────────────────────────────────────────────┘
                                   │
                      ┌────────────┴────────────┐
                      ▼                         ▼
              can_answer = true          can_answer = false
                      │                         │
                      ▼                         ▼
                return answer           4. ITERATE — go back
                                        to SELECT, with the
                                        visited section excluded.
```

Notice what's **not** in this picture: no chunker, no embedding model, no vector store, no top-k. The "index" is a markdown outline of the document with one summary per section, generated once offline. The "retriever" is the LLM itself, reading that outline and choosing where to look.

This only works because two things are already true about the input document:

1. It's a **`DoclingDocument`** — a real tree with sections, paragraphs, tables, and figures, parsed by Docling from the source PDF.
2. Each section already has a **summary attached as metadata**, written by `DoclingEnrichingAgent` in a one-time enrichment pass.

We'll use a fixture that already has both. If you want to run this on your own PDF, there's an optional cell a few steps down that shows you how.

## Loading a pre-enriched document

We'll work with the **Docling tech report** (`arXiv:2408.09869v5`) — a 10-page academic paper with ~20 well-defined sections covering Docling's architecture, AI models, performance benchmarks, and applications. It ships pre-enriched as a fixture in `docling-agent`'s test data, so every section header already carries an LLM-written summary in its metadata.

> **Want to use your own PDF?** Skip ahead to the optional **"Enriching your own PDF"** cell below. Just be aware that enrichment runs an LLM call per section and takes several minutes on a long document.

In [ ]:
from urllib.request import urlretrieve

FIXTURES = Path("fixtures")
FIXTURES.mkdir(exist_ok=True)

fixture_path = FIXTURES / "2408.09869v5-hierarchical-with-summaries.json"
fixture_url = (
    "https://github.com/ibm-granite-community/docling-workshop/"
    "releases/download/fixtures/v1/"
    "2408.09869v5-hierarchical-with-summaries.json"
)
if not fixture_path.exists():
    print(f"Downloading fixture to {fixture_path} ...")
    urlretrieve(fixture_url, fixture_path)

doc = DoclingDocument.load_from_json(fixture_path)
print(f"Loaded document: {doc.name!r}")
print(f"Total items in tree: {sum(1 for _ in doc.iterate_items())}")

### Optional: enriching your own PDF

The cell below shows how to start from a raw PDF instead of the shipped fixture. **Skip it unless you're adapting the notebook to your own document** — it parses a PDF with Docling, then runs `DoclingEnrichingAgent` to attach a summary to every section, which makes one LLM call per section and can take several minutes on a long paper.

In [ ]:
# --- Optional: enrich your own PDF ---
# Uncomment and edit MY_PDF to point at a real PDF on your machine.
# Then re-run this cell instead of the previous "load fixture" cell.

# from docling.document_converter import DocumentConverter
#
# MY_PDF = Path("/path/to/your/document.pdf")
#
# converter = DocumentConverter()
# parsed = converter.convert(str(MY_PDF)).document
#
# enricher = DoclingEnrichingAgent(model_id=MODEL_ID, tools=[])
# doc = enricher.run(
#     task="Summarize each section header.",
#     document=parsed,
# )
# print(f"Enriched document: {doc.name!r}")

## What the "index" actually looks like

Before we run any queries, let's look at the entire retrieval index. In a vector RAG system this would be tens of thousands of float32 vectors in a database. Here it's a single string: a markdown outline of the document, with each section's summary inlined.

In [ ]:
outline = create_document_outline(doc, format=OutlineFormat.MARKDOWN)
print(outline)

That's it. That's the entire retrieval state. No vectors, no nearest-neighbor index, no embedding model. A single markdown blob, generated once when the document was enriched, that fits comfortably in any LLM's context window.

The agent's loop will hand this outline to the model, ask it to pick a section, then fetch *only that section's text* before attempting an answer. The full document is never loaded into the prompt.

## Query 1 — the happy path

Let's start with a question whose answer lives entirely in one section. We'd expect the agent to pick that section on its first try and answer immediately.

In [ ]:
agent = DoclingRAGAgent(
    model_id=MODEL_ID,
    tools=[],
    max_iterations=5,
    verbose=True,
)

query_1 = "What are the main AI models used in Docling?"
answer_doc = agent.run(task=query_1, document=doc)

Walk through what just happened, top to bottom:

1. **Setup panel** — the agent counted available section refs and noted the iteration budget.
2. **Section Selection** — the LLM read the outline (with summaries), picked exactly one section ref, and gave its reason in plain English.
3. **Section Content** — the agent fetched only that section's subtree from the `DoclingDocument`. The full document was never put in the prompt.
4. **Answer Attempt** — the LLM read the section text and decided it had enough to answer. `can_answer = true`.
5. **Final Answer** — the loop converged in **one iteration**.

This is the system on its best behavior. The next query is harder.

## Query 2 — where chunkless RAG actually earns its keep

Query 1 was easy. The document is 10 pages; every section is rich and self-contained; the summaries are precise enough that the right section is the first pick. On a document like that, *any* retrieval method (embedding-based, BM25, random guess) would probably work.

To see where chunkless RAG actually earns its keep, we need a bigger, messier document — the kind a real RAG system deals with. Something with hundreds of sections, stub page-headers, repeated layout fragments, and answers that live two levels deeper than the obvious first pick.

So we swap in IBM's 2025 annual report (the enriched version we prepared earlier with concise topical summaries). Same agent, same loop, same model — different document.

In [ ]:
big_doc_path = FIXTURES / "ibm_annual_report_enriched_concise.json"
big_doc_url = (
    "https://github.com/ibm-granite-community/docling-workshop/"
    "releases/download/fixtures/v1/"
    "ibm_annual_report_enriched_concise.json"
)
if not big_doc_path.exists():
    print(f"Downloading fixture to {big_doc_path} (9.5 MB) ...")
    urlretrieve(big_doc_url, big_doc_path)

big_doc = DoclingDocument.load_from_json(big_doc_path)

small_sections = sum(1 for t in doc.texts if t.label == 'section_header')
big_sections = sum(1 for t in big_doc.texts if t.label == 'section_header')

small_outline = create_document_outline(doc, format=OutlineFormat.MARKDOWN)
big_outline = create_document_outline(big_doc, format=OutlineFormat.MARKDOWN)

print(f"Docling tech report : {small_sections:>3} sections, outline {len(small_outline):>7,} chars")
print(f"IBM annual report   : {big_sections:>3} sections, outline {len(big_outline):>7,} chars")

In [ ]:
query_2 = "What was Red Hat's revenue growth in 2025 and how did it contribute to IBM's overall software segment?"

if os.environ.get("CI"):
    # CI: run the SELECT -> FETCH -> ATTEMPT -> REJECT -> SELECT loop with a tighter
    # iteration budget so the daily workflow finishes in bounded time. Two iterations
    # is enough to exercise the stub-header recovery path (iter 1 hits the stub,
    # iter 2 picks a real section).
    ci_agent = DoclingRAGAgent(model_id=MODEL_ID, tools=[], max_iterations=2, verbose=False)
    answer_doc_2 = ci_agent.run(task=query_2, document=big_doc)
else:
    answer_doc_2 = agent.run(task=query_2, document=big_doc)

The trace tells a different story this time.

**Iteration 1** — the agent picks `#/texts/162`, a section headed "Software". In the outline that entry has no summary — it's a bare heading. When the agent fetches content, it gets back 8 characters: the word "Software." That's because `#/texts/162` is a stub section in the source PDF — a page-level label, not a prose section. The agent reads what it got, sees it's nowhere near enough, and marks `can_answer = false`.

**Iteration 2** — with the stub now excluded from `unvisited`, the agent consults the remaining outline. This time the best match isn't by heading but by summary: `#/texts/167` has the unassuming heading "Management Discussion," but its summary reads *"IBM Software Segment Revenue Performance in 2025."* The agent picks it, fetches ~2,300 characters of actual prose — Red Hat's revenue growth, the segment's percentage contribution — and answers.

Two things worth calling out:

- **The agent navigated by summaries, not by keywords.** The obvious heading-match ("Software") was a dead end; the useful section had an opaque heading but a descriptive summary. In a pure keyword-search or embedding-similarity world, the stub would have won on name match.
- **The agent admitted insufficiency.** When iter 1 came back with 8 chars, the answer step was honest — `can_answer = false` — rather than fabricating something plausible from that thin context. That honesty is what makes the iteration loop useful; without it, the agent would have stopped at the wrong section.

This is the core multi-hop mechanic of chunkless RAG: read summaries, pick, read content, reconsider, pick again. No embeddings, no vector index, no cosine similarity scores — just the document's structure and the model's judgment.

## Query 3 — what happens when the answer isn't in the document

Now a question the document genuinely can't answer. Watch what the agent does.

In [ ]:
query_3 = "What is the price of a Docling enterprise license?"
answer_doc_3 = agent.run(task=query_3, document=doc)

Notice three things about that trace:

1. **The agent did iterate.** It tried multiple sections in good faith — "maybe there's a pricing section," "maybe there's a licensing section," "maybe there's a commercial offering section."
2. **It ran out the clock.** When it hit `max_iterations` without converging, it returned a `Partial Answer` rather than guessing.
3. **It did not invent a number.** This is the single most important property of chunkless RAG for high-stakes use cases: there is no "top-k of vaguely-related chunks" to seduce the model into producing a confident-sounding but ungrounded answer. Either a section actually contains the information, or it doesn't.

(If your model *did* fabricate an answer here, that's a property of the model, not the retrieval system — chunkless RAG can't cure dishonest models, it just gives them less to riff on.)

## Side-by-side: chunkless vs. hybrid chunking

Both approaches can answer Query 1. They produce differently-shaped answers, and the difference is the whole point of this tutorial.

To make the comparison concrete, let's run a minimal hybrid-chunking RAG pipeline against the same tech report, with the same model, and look at what comes out. We'll use `HybridChunker` from `docling-core` to break the document into chunks, rank them by lexical overlap with the query (a lightweight stand-in for vector search — a production pipeline would use embeddings), and feed the top 3 to Granite as context.


In [ ]:
import re

from docling_core.transforms.chunker.hybrid_chunker import HybridChunker
from mellea import MelleaSession
from mellea.stdlib.components import Message
from mellea.stdlib.context import ChatContext

# Chunk the same tech report the chunkless agent used for Query 1.
chunker = HybridChunker()
chunks = list(chunker.chunk(doc))
print(f"HybridChunker produced {len(chunks)} chunks.")

# Rank by lexical overlap with the query. In a real pipeline you'd use an
# embedding model (e.g. ibm-granite/granite-embedding-107m-multilingual) and
# cosine similarity; we keep the dependency footprint small here.
def _tok(s: str) -> set[str]:
    return set(re.findall(r"[a-z]{3,}", s.lower()))

q_tokens = _tok(query_1)
top3 = sorted(chunks, key=lambda c: len(q_tokens & _tok(chunker.contextualize(c))), reverse=True)[:3]

for i, ch in enumerate(top3, 1):
    text = chunker.contextualize(ch)
    print(f"\n--- top chunk {i} ({len(text)} chars) ---\n{text[:300]}{'...' if len(text) > 300 else ''}")

# Feed the retrieved chunks to the same Granite 4.0 model the chunkless
# agent used, through the same monkey-patched backend.
ctx = ChatContext().add(Message(
    role="system",
    content=(
        "You are a helpful assistant. Answer the user's question using only "
        "the provided context. If the answer is not in the context, say so."
    ),
))
hybrid_session = MelleaSession(ctx=ctx, backend=_make_backend(MODEL_ID))

context_blob = "\n\n---\n\n".join(chunker.contextualize(ch) for ch in top3)
prompt = f"Context:\n{context_blob}\n\nQuestion: {query_1}\n\nAnswer:"

hybrid_answer = hybrid_session.instruct(prompt).value
print("\n=== Hybrid-chunking RAG answer ===\n")
print(hybrid_answer)


Look at the *shape* of the two answers.

**Hybrid-chunking RAG answered:**

> Based on the provided context, the main AI models used in Docling are:
>
> 1. Specialized AI models for layout analysis
> 2. Specialized AI models for table structure recognition
>
> The context mentions that Docling builds on the powerful, specialized AI models and datasets for layout analysis and table structure recognition that were developed and presented in recent work by the authors [12, 13, 9]. However, it does not provide specific details on the exact AI model architectures or names used.

**Chunkless `DoclingRAGAgent` answered:**

> The main AI models used in Docling are: 1) a layout analysis model – an accurate object‑detector for page elements (based on an RT‑DETR architecture) that predicts bounding boxes and classes for elements such as text, figures, tables, etc.; and 2) TableFormer, a state‑of‑the‑art vision‑transformer model for recognizing the logical row and column structure of tables, including handling merged cells, spans, and hierarchy. Both models are released with pre‑trained weights on Hugging Face and are provided via the docling‑ibm‑models package.

The hybrid answer is an **extraction**: it returns what its top-ranked chunks literally contained, lightly stitched together. The top-3 chunks on this query were the *introduction*, the *processing pipeline overview*, and the *PDF backends* subsection — all of which mention AI models in passing but don't actually name them. Section 3.2 ("AI models"), which names RT-DETR and TableFormer, didn't make the top 3 on lexical score. The answer is hedgy because the evidence is: the model correctly refused to invent names that weren't in its context.

The chunkless answer is a **synthesis**: the agent reasoned over the outline, picked the one section that was explicitly about the question (`#/texts/49`, titled "AI models"), read the whole section at once, and produced a grounded, specific answer on the first iteration. No chunk boundary cut through the subsection structure, so the model saw RT-DETR and TableFormer described together the way they were written.

Neither is universally better. If you want the exact sentence where a keyword appears, hybrid can do that cheaply. If you want an answer grounded in the author's original section structure — including lists, sub-headings, and tables that chunking would fragment — chunkless wins. They're answering subtly different questions, and the right choice depends on what your reader actually needs.

## When to reach for which

The honest version of this section is: **chunkless and hybrid chunking are not really competitors.** They sit at different granularities of the same Docling-parsed tree. Hybrid chunking indexes leaf nodes; chunkless navigates section nodes. Use whichever fits your problem.

| | Hybrid chunking + vector store | Chunkless `DoclingRAGAgent` |
| --- | --- | --- |
| **Best for** | Large corpora, fact lookup | Single long structured docs, section-level Q&A |
| **Retrieval unit** | Paragraph / table / leaf node | Whole section subtree |
| **Index** | Embeddings over chunks | Markdown outline + per-section summaries |
| **Build cost** | One embedding pass | One LLM summarization pass (slower, $$) |
| **Query latency** | Milliseconds | Seconds (LLM in the loop) |
| **Scales to corpus size** | Excellent (millions of docs) | Poor (tens of docs at most) |
| **Cross-document synthesis** | Native (top-k mixes freely) | Per-doc loops merged at the end |
| **Citation granularity** | Down to a paragraph or table cell | Down to a section ref |
| **Audit trail** | Cosine scores | Plain-English "I picked this section because..." |
| **Failure mode** | Returns unrelated chunks | Wastes an iteration, falls back gracefully |

**Rule of thumb:** if your reader's question is *"find the sentence that says X,"* reach for hybrid chunking. If it's *"explain how the document handles X,"* reach for chunkless. And if you're not sure — try chunkless first on a single representative document. It needs no extra infrastructure, the verbose trace tells you exactly what it did, and you can swap to hybrid chunking later if you outgrow it.

## Where to go next

**Read the source.** The whole `DoclingRAGAgent` is roughly 450 lines in `docling_agent/agent/rag.py`. If you want to see how `_select_section` prompts the model, how `_attempt_answer` validates JSON output with Mellea's `RejectionSamplingStrategy`, or how `_collect_flat_section_text` handles flat (non-hierarchical) documents, it's all there. There's no hidden machinery.

**Knobs you can turn.** Two parameters do most of the work:

- `max_iterations` — how many sections the agent is allowed to consult before giving up. Default 5. Lower it for tight latency budgets, raise it for harder questions over longer documents.
- `verbose` — turns the rich-console trace on or off. Set it to `False` in production. The structured iteration history is still available on the `RAGResult` object if you need to log or audit it.

**What we didn't cover.** `DoclingRAGAgent` also supports passing multiple documents through `sources=[...]`. It runs the loop once per document and then synthesizes a final answer from the per-document partials via `_merge_answers`. That's a natural follow-up if you're building something that needs to span more than one paper at a time.

**The bigger picture.** The reason chunkless RAG works at all is that Docling has already done the structural parsing. Every approach in this notebook — outline navigation, per-section summaries, subtree fetching — is downstream of "Docling parsed a real tree out of this PDF." If you're working with documents that *don't* have real structure (chat logs, OCR'd forms, scanned receipts), this technique falls back to "return the whole document," and hybrid chunking is the better tool for that job.